In [2]:
import kagglehub
import os
import pandas as pd
from datasets import load_dataset
from kagglehub import KaggleDatasetAdapter

In [8]:
def download_dataset()->list[str]:
    """
    Download the dataset from Kaggle and return the paths to the files.
    """
    dataset_dir = kagglehub.dataset_download("josephleake/huge-collection-of-reddit-votes")
    paths = []
    for dir_path, _, file_names in os.walk(dataset_dir):
        for file_name in file_names:
            paths.append(os.path.join(dir_path, file_name))
    print(f'File path to votes:\n{paths[0]}')
    print(f'File path to submissions:\n{paths[1]}')
    return paths

def get_dataframe()->tuple[pd.DataFrame]:
    """
    Return a tuple of two pandas.Dataframe: votes and submissions.

    Returns:
        tuple[pd.DataFrame]: a tuple of two dataframes.
    """
    paths = download_dataset()
    votes = pd.read_csv(paths[0], sep='\t')
    submissions = pd.read_csv(paths[1], sep='\t')
    return (votes, submissions)

def view_users_votes(votes:pd.DataFrame):
    view = (
        votes
        .groupby(['USERNAME', 'SUBREDDIT', 'VOTE'])
        .size()                         # count upvotes/downvotes in each group
        .unstack(fill_value=0)          # pivot VOTE labels into columns
        .rename(columns={
            'upvote':   'num_upvotes',
            'downvote': 'num_downvotes'
        })
        .reset_index()                  # turn USERNAME & SUBREDDIT back into columns
    )
    return view

In [ ]:
votes, submissions = get_dataframe()

In [6]:
votes[400:500]

,SUBMISSION_ID,SUBREDDIT,CREATED_TIME,USERNAME,VOTE
400,t3_dyudhy,r/13or30,NaN,-Rick_Sanchez_,upvote
401,t3_dzmkff,r/KidsAreFuckingStupid,NaN,-Rick_Sanchez_,upvote
402,t3_dz14rj,r/maybemaybemaybe,NaN,-Rick_Sanchez_,upvote
403,t3_dynyrs,r/classicwow,NaN,-Rick_Sanchez_,upvote
404,t3_dyl1v4,r/MyPeopleNeedMe,NaN,-Rick_Sanchez_,upvote
...,...,...,...,...,...
495,t3_dyooji,r/gifs,NaN,-Rick_Sanchez_,upvote
496,t3_dy0o9m,r/aww,NaN,-Rick_Sanchez_,upvote
497,t3_dym6dw,r/dogswithjobs,NaN,-Rick_Sanchez_,upvote
498,t3_dz43xt,r/therewasanattempt,NaN,-Rick_Sanchez_,upvote


In [9]:
user_votes_view = view_users_votes(votes)

In [10]:
user_votes_view.head(100)

VOTE,USERNAME,SUBREDDIT,num_downvotes,num_upvotes
0,---UUU---,r/Artistic_Hentai,5,2
1,---UUU---,r/AsianNSFW,1,0
2,---UUU---,r/BikiniBottomTwitter,0,1
3,---UUU---,r/CelebEconomy,0,10
4,---UUU---,r/ClashOfClans,16,3
...,...,...,...,...
95,--NiNjA--,r/GalaxyS6,1,0
96,--NiNjA--,r/GalaxyWatch,1,13
97,--NiNjA--,r/Gamingcirclejerk,2,0
98,--NiNjA--,r/GetMotivated,4,7


In [ ]:
def filter_subreddits(
    votes: pd.DataFrame,
    num_upvotes: int = 0,
    num_downvotes: int = 0,
    total_votes: int = 0,
    num_users: int = 0,
) -> pd.DataFrame:
    """
    Filter a DataFrame of subreddit vote counts according to given thresholds.

    Parameters:
    - votes: DataFrame with at least ['USERNAME', 'SUBREDDIT', 'num_upvotes', 'num_downvotes'] columns.
    - num_upvotes: keep rows where num_upvotes > this value (if > 0).
    - num_downvotes: keep rows where num_downvotes > this value (if > 0).
    - total_votes: keep rows where (num_upvotes + num_downvotes) > this value (if > 0).
    - num_users: keep rows where the subreddit has more than this many unique users (if > 0).

    Returns:
    - Filtered DataFrame.
    """
    # Start with an all-True mask
    mask = pd.Series(True, index=votes.index)

    # Apply upvotes threshold
    if num_upvotes > 0:
        mask &= votes['num_upvotes'] > num_upvotes

    # Apply downvotes threshold
    if num_downvotes > 0:
        mask &= votes['num_downvotes'] > num_downvotes

    # Apply total votes threshold
    if total_votes > 0:
        mask &= (votes['num_upvotes'] + votes['num_downvotes']) > total_votes

    # Apply distinct user count per subreddit threshold
    if num_users > 0:
        # Compute number of unique users for each subreddit
        user_counts = votes.groupby('SUBREDDIT')['USERNAME'].transform('nunique')
        mask &= user_counts > num_users

    # Return a copy of the filtered DataFrame
    return votes[mask].copy()

In [56]:
subreddits = filter_subreddits(user_votes_view, 0, 0, 100, 200)
subreddits['SUBREDDIT'].unique().shape

(2366,)